# 8.6. Residual Networks \(ResNet\) and ResNeXt

[Last chapter](https://github.com/DonaldKellett/my-ascend-notebooks/blob/0ede0919c0edc08eeae1acc502e4fc33a57cc2d3/orangepiaipro-20t/13-d2l-mindspore-ch8-modern-convolutional-neural-networks/00-batch-normalization.ipynb) we saw what batch normalization is, how it works and when to use it. This time, let's study _the_ standard for CNNs in modern deep learning - ResNet.

This notebook is based on the [AtomGit AI Notebook Lab](https://ai.gitcode.com/docs/notebooks/free-usage/) cloud environment providing datacenter-level Ascend 910B4 training-optimized NPUs. The software used in this notebook are listed below.

1. Ubuntu 22.04 LTS
1. Python 3.11
1. MindSpore 2.8.0
1. CANN 8.5.0

In [1]:
!npu-smi info

+------------------------------------------------------------------------------------------------+
| npu-smi 25.5.1                   Version: 25.5.1                                               |
+---------------------------+---------------+----------------------------------------------------+
| NPU   Name                | Health        | Power(W)    Temp(C)           Hugepages-Usage(page)|
| Chip                      | Bus-Id        | AICore(%)   Memory-Usage(MB)  HBM-Usage(MB)        |
+===========================+===============+====================================================+
| 2     910B4               | OK            | 88.2        43                0    / 0             |
| 0                         | 0000:C2:00.0  | 0           0    / 0          2870 / 32768         |
+===========================+===============+====================================================+
+---------------------------+---------------+----------------------------------------------------+
| NPU     

In [2]:
%pip install mindspore==2.8.0 \
    -i https://repo.mindspore.cn/pypi/simple \
    --trusted-host repo.mindspore.cn \
    --extra-index-url https://repo.huaweicloud.com/repository/pypi/simple

Defaulting to user installation because normal site-packages is not writeable
Looking in indexes: https://repo.mindspore.cn/pypi/simple, https://repo.huaweicloud.com/repository/pypi/simple

[notice] A new release of pip is available: 25.3 -> 26.1.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [3]:
import mindspore

mindspore.set_device(device_target='Ascend', device_id=0)
mindspore.run_check()

/usr/local/python3.11.14/lib/python3.11/site-packages/numpy/core/getlimits.py:549: UserWarning: The value of the smallest subnormal for <class 'numpy.float32'> type is zero.
  setattr(self, word, getattr(machar, word).flat[0])
/usr/local/python3.11.14/lib/python3.11/site-packages/numpy/core/getlimits.py:89: UserWarning: The value of the smallest subnormal for <class 'numpy.float32'> type is zero.
  return self._float_to_str(self.smallest_subnormal)
/usr/local/python3.11.14/lib/python3.11/site-packages/numpy/core/getlimits.py:549: UserWarning: The value of the smallest subnormal for <class 'numpy.float64'> type is zero.
  setattr(self, word, getattr(machar, word).flat[0])
/usr/local/python3.11.14/lib/python3.11/site-packages/numpy/core/getlimits.py:89: UserWarning: The value of the smallest subnormal for <class 'numpy.float64'> type is zero.
  return self._float_to_str(self.smallest_subnormal)


MindSpore version:  2.8.0


[WARNING] DEVICE(5679,ffff1ebef120,python3.11):2026-05-19-22:33:36.409.434 [mindspore/ccsrc/plugin/ascend/res_manager/mem_manager/ascend_vmm_adapter.h:176] CheckVmmDriverVersion] Open file /etc/ascend_install.info failed.
[WARNING] DEVICE(5679,ffff1ebef120,python3.11):2026-05-19-22:33:36.409.482 [mindspore/ccsrc/plugin/ascend/res_manager/mem_manager/ascend_vmm_adapter.h:204] CheckVmmDriverVersion] Open file /usr/local/Ascend/driver/version.info failed.


The result of multiplication calculation is correct, MindSpore has been installed on platform [Ascend] successfully!


## 8.6.2. Residual Blocks

Each residual block in ResNet is divided into 2 branches.

1. A series of convolutions, batch normalization layers and ReLU activation
    1. $3 \times 3$ convolution with `1px` padding in all directions. Optionally, the number of output channels can be different from the number of input channels. Additionally, the stride can be set larger than 1 for downsampling
    1. A batch normalization layer that comes _before_ the ReLU activation
    1. ReLU activation layer
    1. $3 \times 3$ convolution with `1px` padding in all directions. The number of channels and feature map resolution is preserved
    1. Batch normalization layer
1. An optional $1 \times 1$ convolution layer for changing the number of channels and resolution of the input image or feature map. If the number of channels and resolution remains unchanged then no layer is required and we simply use the identity mapping

The sum of outputs from both branches are computed before passing the result to the final ReLU activation layer. Note that this is _different_ from the Inception block from GoogLeNet which concatenates the output from all branches along the channel dimension.

The idea of the residual block is to allow it to more easily learn the identity mapping $f(\mathbf{x}) = \mathbf{x}$ through the second branch. This ensures that as we add more layers to our deep network, it becomes _strictly_ more expressive rather than just different, allowing it to learn better from the given training data and producing more accurate predictions.

Our implementation of the residual block from ResNet with MindSpore 2.8.0 as below.

In [4]:
import mindspore.nn as nn

class Residual(nn.Cell):
    def __init__(self, in_channels, out_channels, stride=1):
        super().__init__()
        # Whether to use 1x1 convolution
        self.use_1x1conv = in_channels != out_channels or stride != 1
        self.b1 = nn.SequentialCell([
            nn.Conv2d(in_channels, out_channels, kernel_size=3, stride=stride),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(),
            nn.Conv2d(out_channels, out_channels, kernel_size=3),
            nn.BatchNorm2d(out_channels)
        ])
        b2 = []
        if self.use_1x1conv:
            b2.append(nn.Conv2d(in_channels, out_channels, kernel_size=1, stride=stride))
        self.b2 = nn.SequentialCell(b2)
        self.relu = nn.ReLU()

    def construct(self, X):
        return self.relu(self.b1(X) + self.b2(X))

Let's try it with and without the $1 \times 1$ convolution in the second branch, inspect the output shapes produced by our residual block.

In [5]:
import mindspore.ops as ops

residual_without_1x1conv = Residual(3, 3)
X = ops.rand((4, 3, 6, 6))
residual_without_1x1conv(X).shape

(4, 3, 6, 6)

In [6]:
residual_with_1x1conv = Residual(3, 6, stride=2)
residual_with_1x1conv(X).shape

(4, 6, 3, 3)

## 8.6.3. ResNet Model

ResNet comes with multiple variations depending on the total number of convolutional + FC layers. The variation we'll study this chapter is ResNet-18.

ResNet-18 is comprised of 17 convolutional layers plus a final FC layer. Like GoogLeNet, it has a clear stem/body/head structure.

The body consists of 4 modules. Each module consists of 2 residual blocks. Except for the first module, the remaining modules double the number of channels and halves the feature map resolution for downsampling.

Each module is described below in sequence. To simplify our illustration, we'll treat the stem and head as separate modules as well.

1. Stem module
    1. $7 \times 7$ convolution with a stride of 2 for downsampling. The number of channels is increased to 64
    1. Batch normalization before the ReLU activation
    1. ReLU activation layer
    1. Max pooling layer with a $3 \times 3$ pooling window and stride of 2 for downsampling
1. 1st module in body consisting of 2 residual blocks preserving the number of channels and feature map resolution
1. 2nd module in body consisting of 2 residual blocks
    1. The 1st residual block doubles the number of channels to 128 and halves the feature map resolution
    1. The 2nd residual block preserves the channel dimensionality and resolution
1. 3rd module in body similar to \(3\). The number of channels doubled to 256 and resolution halved
1. 4th module in body similar to \(4\). The number of channels doubled to 512 and resolution halved
1. Head module
    1. Global average pooling layer reducing resolution to $1 \times 1$
    1. Flattening layer
    1. Final FC layer yielding logits corresponding to class probabilities

Our implementation with MindSpore 2.8.0 below.

In [7]:
resnet18 = nn.SequentialCell([
    # Stem module
    nn.SequentialCell([
        nn.Conv2d(1, 64, kernel_size=7, stride=2),
        nn.BatchNorm2d(64),
        nn.ReLU(),
        nn.MaxPool2d(kernel_size=3, stride=2, pad_mode='same')
    ]),
    # 1st module in body
    nn.SequentialCell([
        Residual(64, 64),
        Residual(64, 64)
    ]),
    # 2nd module in body
    nn.SequentialCell([
        Residual(64, 128, stride=2),
        Residual(128, 128)
    ]),
    # 3rd module in body
    nn.SequentialCell([
        Residual(128, 256, stride=2),
        Residual(256, 256)
    ]),
    # 4th module in body
    nn.SequentialCell([
        Residual(256, 512, stride=2),
        Residual(512, 512)
    ]),
    # Head module
    nn.SequentialCell([
        nn.AdaptiveAvgPool2d((1, 1)),
        nn.Flatten(),
        nn.Dense(512, 10)
    ])
])
resnet18

SequentialCell(
  (0): SequentialCell(
    (0): Conv2d(input_channels=1, output_channels=64, kernel_size=(7, 7), stride=(2, 2), pad_mode=same, padding=0, dilation=(1, 1), group=1, has_bias=False, weight_init=<mindspore.common.initializer.HeUniform object at 0xfffe6c8a20d0>, bias_init=None, format=NCHW)
    (1): BatchNorm2d(num_features=64, eps=1e-05, momentum=0.9, gamma=Parameter (name=0.1.gamma, shape=(64,), dtype=Float32, requires_grad=True), beta=Parameter (name=0.1.beta, shape=(64,), dtype=Float32, requires_grad=True), moving_mean=Parameter (name=0.1.moving_mean, shape=(64,), dtype=Float32, requires_grad=False), moving_variance=Parameter (name=0.1.moving_variance, shape=(64,), dtype=Float32, requires_grad=False))
    (2): ReLU()
    (3): MaxPool2d(kernel_size=3, stride=2, pad_mode=SAME)
  )
  (1): SequentialCell(
    (0): Residual(
      (b1): SequentialCell(
        (0): Conv2d(input_channels=64, output_channels=64, kernel_size=(3, 3), stride=(1, 1), pad_mode=same, padding=0, dila

Let's inspect the output shape after each module given an initial grayscale input image with dimensions of $96 \times 96$ pixels.

In [8]:
def layer_summary(net, X_shape):
    print(f'Input shape: {X_shape}')
    X = ops.randn(*X_shape)
    for cell in net.cells():
        X = cell(X)
        print(f'Output shape from {cell.__class__.__name__}: {X.shape}')

X_shape = (1, 1, 96, 96)
layer_summary(net=resnet18, X_shape=X_shape)

Input shape: (1, 1, 96, 96)
Output shape from SequentialCell: (1, 64, 24, 24)
Output shape from SequentialCell: (1, 64, 24, 24)
Output shape from SequentialCell: (1, 128, 12, 12)
Output shape from SequentialCell: (1, 256, 6, 6)
Output shape from SequentialCell: (1, 512, 3, 3)
Output shape from SequentialCell: (1, 10)


## 8.6.4. Training

TODO